In [65]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csc_matrix

In [66]:
df_movies = pd.read_csv('/content/netflix_cleaned_dataset.csv.csv')

In [67]:
print("First 5 rows of the dataset:")
display(df_movies.head())

First 5 rows of the dataset:


,Title,Genre,parent_genre_str,main_parent_genre,metadata,poster,year_premiere,IMDB Score,Runtime,runtime_normalized,primary_language_capitalized,Premiere,Director,Plot
0,enter the anime,documentary,documentary,documentary,movie title is enter the anime. genre is docum...,https://m.media-amazon.com/images/M/MV5BNzljM2...,2019.0,2.5,58,0.263415,English,"August 5, 2019",Alex Burunova,It is a documentary special aimed at newcomers...
1,dark forces,thriller,thriller,thriller,movie title is dark forces. genre is thriller....,https://m.media-amazon.com/images/M/MV5BOTIxMW...,2020.0,2.6,81,0.375610,Spanish,"August 21, 2020",Bernardo Arellano,"In search of his sister, a renegade criminal s..."
2,the app,"science fiction, drama","scifi, drama",scifi,"movie title is the app. genre is scifi, drama....",https://m.media-amazon.com/images/M/MV5BNzgzZG...,2019.0,2.6,79,0.365854,Italian,"December 26, 2019",Elisa Fuksas,"Loving girlfriend, family fortune, breakout mo..."
3,the open house,horror thriller,"horror, thriller",horror,movie title is the open house. genre is horror...,https://m.media-amazon.com/images/M/MV5BMTU0Mj...,2018.0,3.2,94,0.439024,English,"January 19, 2018","Matt Angel, Suzanne Coote",A teenager and his mother find themselves besi...
4,kaali khuhi,mystery,mystery,mystery,movie title is kaali khuhi. genre is mystery. ...,https://m.media-amazon.com/images/M/MV5BZjRhNG...,2020.0,3.4,90,0.419512,Hindi,"October 30, 2020",Terrie Samundra,"Shivangi, a 10 year old girl, is put to the ul..."


In [68]:
num_unique_genres = df_movies['main_parent_genre'].nunique()
print(f"Number of unique genre categories in the dataset: {num_unique_genres}")

Number of unique genre categories in the dataset: 17


In [69]:
unique_genres = df_movies['main_parent_genre'].unique()
print("\nInformasi dataset:")
display(df_movies.info())



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529 entries, 0 to 528
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Title                         529 non-null    object 
 1   Genre                         529 non-null    object 
 2   parent_genre_str              529 non-null    object 
 3   main_parent_genre             529 non-null    object 
 4   metadata                      529 non-null    object 
 5   poster                        529 non-null    object 
 6   year_premiere                 529 non-null    float64
 7   IMDB Score                    529 non-null    float64
 8   Runtime                       529 non-null    int64  
 9   runtime_normalized            529 non-null    float64
 10  primary_language_capitalized  529 non-null    object 
 11  Premiere                      529 non-null    object 
 12  Director                      522 non-nul

None

In [70]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df_movies['main_parent_genre'].fillna(''))
print(f"TF-IDF matrix shape for genres: {tfidf_matrix.shape}")

TF-IDF matrix shape for genres: (529, 16)


In [71]:
tfidf_director = TfidfVectorizer(stop_words='english')
director_matrix = tfidf_director.fit_transform(df_movies['Director'].fillna(''))
print(f"TF-IDF matrix shape for directors: {tfidf_matrix.shape}")

TF-IDF matrix shape for directors: (529, 969)


In [72]:
scaler = MinMaxScaler()

numeric_features_for_fitting = df_movies[['IMDB Score', 'runtime_normalized', 'year_premiere']]
scaled_numeric_features_array = scaler.fit_transform(numeric_features_for_fitting)

imdb_normalized = scaled_numeric_features_array[:, 0].reshape(-1, 1)
runtime_normalized_scaled_for_sparse = scaled_numeric_features_array[:, 1].reshape(-1, 1)
year_normalized = scaled_numeric_features_array[:, 2].reshape(-1, 1)

print(f"MinMaxScaler fitted on {scaler.n_features_in_} features.")

MinMaxScaler fitted on 3 features.


In [73]:
from scipy.sparse import csc_matrix

imdb_sparse = csc_matrix(imdb_normalized)
runtime_sparse = csc_matrix(runtime_normalized_scaled_for_sparse)
year_sparse = csc_matrix(year_normalized)

In [74]:
numeric_features = hstack([imdb_sparse, runtime_sparse, year_sparse])
print(f"Shape of the aligned numerical feature matrix: {numeric_features.shape}")

Shape of the aligned numerical feature matrix: (529, 3)


In [75]:
feature_matrix = hstack([tfidf_matrix, director_matrix, numeric_features])
print(f"Shape of the final feature matrix: {feature_matrix.shape}")

Shape of the final feature matrix: (529, 988)


In [76]:
cosine_sim = cosine_similarity(feature_matrix)
print(f"Shape of the cosine similarity matrix: {cosine_sim.shape}")

Shape of the cosine similarity matrix: (529, 529)


In [77]:
kolom_relevan = ['Title', 'main_parent_genre', 'Director', 'year_premiere', 'IMDB Score', 'Runtime', 'poster', 'primary_language_capitalized', 'Premiere', 'Plot']
df_relevan = df_movies[kolom_relevan].copy()

In [78]:
api_artifacts = {
    'tfidf_vectorizer': tfidf_vectorizer,
    'tfidf_director': tfidf_director,
    'scaler': scaler,
    'feature_matrix': feature_matrix,
    'df_movies': df_relevan
}

joblib.dump(api_artifacts, 'model_rekomendasi.pkl')
print('model_rekomendasi_total.pkl has been successfully saved with all components included.')

model_rekomendasi_total.pkl has been successfully saved with all components included.


In [79]:
import numpy as np

def demonstrasi_rekomendasi_multi_genre(genre_list, director_input=None, year_input=None, top_n=5):
    """
    Demonstration function that supports 1 to 3 genre inputs at once (as a list),
    with optional director and premiere year inputs.
    """
    # Ensure genre input is a list; convert single string input automatically
    if isinstance(genre_list, str):
        genre_list = [genre_list]

    # 1. Combine the genre list into a single space-separated string
    # Example: ['action', 'thriller', 'comedy'] -> 'action thriller comedy'
    genre_combined_string = " ".join(genre_list)

    # Transform genre input using the combined string
    user_genre_vector = tfidf_vectorizer.transform([genre_combined_string])

    # 2. Transform director input (optional)
    if director_input:
        user_director_vector = tfidf_director.transform([director_input])
    else:
        user_director_vector = csc_matrix((1, tfidf_director.transform(['']).shape[1]))

    # 3. Transform year input (optional) and numerical features
    default_imdb = df_movies['IMDB Score'].median()
    default_runtime = df_movies['runtime_normalized'].median()

    # Use input year or median value if not provided
    chosen_year = year_input if year_input else df_movies['year_premiere'].median()

    # Transform user input for numerical features using the fitted scaler
    # The order must match the order used during fitting:
    # 'IMDB Score', 'runtime_normalized', 'year_premiere'
    user_numeric_input_for_scaling = [[default_imdb, default_runtime, chosen_year]]
    user_numeric_scaled = scaler.transform(user_numeric_input_for_scaling)

    user_imdb_scaled = user_numeric_scaled[0][0]
    user_runtime_scaled = user_numeric_scaled[0][1]
    user_year_scaled = user_numeric_scaled[0][2]

    # Wrap numerical features into a sparse matrix
    user_numeric_vector = hstack([
        csc_matrix([[user_imdb_scaled]]),
        csc_matrix([[user_runtime_scaled]]),
        csc_matrix([[user_year_scaled]])
    ])

    # 4. Combine all user input vectors into a single feature vector
    user_feature_vector = hstack([user_genre_vector, user_director_vector, user_numeric_vector])

    # 5. Compute cosine similarity between the user input and all movies in the dataset
    similarity_scores = cosine_similarity(user_feature_vector, feature_matrix).flatten()

    # 6. Get indices of the highest-scoring movies
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    # 7. Display recommendation results
    print(f"\n=== MULTI-GENRE RECOMMENDATION RESULTS ===")
    print(f"User Input -> Selected Genres : {genre_list}")
    print(f"              Director        : {director_input}")
    print(f"              Premiere Year   : {year_input}\n")

    hasil = df_relevan.iloc[top_indices].copy()
    hasil['Similarity Score'] = similarity_scores[top_indices]

    display(hasil[['Title', 'main_parent_genre', 'Director', 'year_premiere', 'IMDB Score', 'Plot', 'Similarity Score']])

### Demonstrasi 1: Rekomendasi berdasarkan 3 genre favorit

In [80]:
demonstrasi_rekomendasi_multi_genre(
    genre_list=["action", "thriller", "comedy"],
    director_input=None,
    year_input=None
)


=== MULTI-GENRE RECOMMENDATION RESULTS ===
User Input -> Selected Genres : ['action', 'thriller', 'comedy']
              Director        : None
              Premiere Year   : None



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


,Title,main_parent_genre,Director,year_premiere,IMDB Score,Plot,Similarity Score
361,the old guard,action,Gina Prince-Bythewood,2020.0,6.7,A covert group of tight-knit mercenaries with ...,0.745614
348,extraction,action,Sam Hargrave,2020.0,6.7,"Tyler Rake, a fearless black market mercenary,...",0.743980
199,project power,action,"Henry Joost, Ariel Schulman",2020.0,6.0,When a pill that gives its users unpredictable...,0.732995
259,polar,action,Jonas Åkerlund,2019.0,6.3,A retiring assassin suddenly finds himself on ...,0.724654
206,6 underground,action,Michael Bay,2019.0,6.1,It follows a group of people who fake their de...,0.723809


### Demonstrasi 2: Rekomendasi berdasarkan 1 genre dan director favorit

In [81]:
demonstrasi_rekomendasi_multi_genre(
    genre_list=["drama"],
    director_input="Steven Spielberg",
    year_input=None
)


=== MULTI-GENRE RECOMMENDATION RESULTS ===
User Input -> Selected Genres : ['drama']
              Director        : Steven Spielberg
              Premiere Year   : None



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


,Title,main_parent_genre,Director,year_premiere,IMDB Score,Plot,Similarity Score
235,high flying bird,drama,Steven Soderbergh,2019.0,6.2,"During a pro basketball lockout, a sports agen...",0.882362
453,el camino: a breaking bad movie,drama,NaN,2019.0,7.3,"A look behind the scenes of El Camino (2019), ...",0.815606
268,the outsider,drama,NaN,2018.0,6.3,When an insidious supernatural force edges its...,0.813775
315,monster,drama,NaN,2021.0,6.5,Chronicles the real life stories of infamous k...,0.811079
157,barry,drama,NaN,2016.0,5.8,A hit man from the Midwest moves to Los Angele...,0.784176


### Demonstrasi 3: Rekomendasi berdasarkan 2 genre dan tahun film favorit

In [82]:
demonstrasi_rekomendasi_multi_genre(
    genre_list=["horror", "mystery"],
    director_input=None,
    year_input=2005
)


=== MULTI-GENRE RECOMMENDATION RESULTS ===
User Input -> Selected Genres : ['horror', 'mystery']
              Director        : None
              Premiere Year   : 2005



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


,Title,main_parent_genre,Director,year_premiere,IMDB Score,Plot,Similarity Score
29,i am the pretty thing that lives in the house,horror,Osgood Perkins,2016.0,4.6,A nervous nurse who scares easily finds hersel...,0.214201
310,gerald's game,horror,Mike Flanagan,2017.0,6.5,A couple tries to spice up their marriage in a...,0.202804
18,death note,horror,NaN,2017.0,4.4,An intelligent high school student goes on a s...,0.188363
284,my own man,documentary,David Sampliner,2014.0,6.4,When a man discovers he will be the father to ...,0.184202
502,beasts of no nation,drama,Cary Joji Fukunaga,2015.0,7.7,"A drama based on the experiences of Agu, a chi...",0.183645


### Demonstrasi 4: Rekomendasi dengan semua kriteria (multiple genres, director, year)

In [83]:
demonstrasi_rekomendasi_multi_genre(
    genre_list=["action", "scifi"],
    director_input="Quentin Tarantino",
    year_input=1994
)


=== MULTI-GENRE RECOMMENDATION RESULTS ===
User Input -> Selected Genres : ['action', 'scifi']
              Director        : Quentin Tarantino
              Premiere Year   : 1994



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


,Title,main_parent_genre,Director,year_premiere,IMDB Score,Plot,Similarity Score
284,my own man,documentary,David Sampliner,2014.0,6.4,When a man discovers he will be the father to ...,0.105751
262,spectral,scifi,Nic Mathieu,2016.0,6.3,A sci-fi/thriller story centered on a special-...,0.085033
275,arq,scifi,Tony Elliott,2016.0,6.4,"Trapped in a lab and stuck in a time loop, a d...",0.079181
502,beasts of no nation,drama,Cary Joji Fukunaga,2015.0,7.7,"A drama based on the experiences of Agu, a chi...",0.064350
525,winter on fire: ukraine's fight for freedom,documentary,Evgeny Afineevsky,2015.0,8.4,A documentary on the unrest in Ukraine during ...,0.058128
